In [ ]:
from collections import Counter
from pathlib import Path

folder = Path("dataset/dataset/malicious")

extensions = Counter(
    p.suffix.lower() or "<no extension>"
    for p in folder.iterdir()
    if p.is_file()
)

extensions

Counter({'.wav': 10929, '.mp3': 40})

In [ ]:
import os
import torch
import librosa
import numpy as np
from torch.utils.data import Dataset, DataLoader, random_split
import torch.nn as nn
import torch.optim as optim

class SpectrogramDataset(Dataset):
    def __init__(self, base_path, sr=16000, window_size=5.0, min_overlap=1.5): #sr=sample rate
        self.sr = sr
        self.window_size = int(window_size * sr) #we convert time into array indices
        self.min_overlap = int(min_overlap * sr)
        self.samples = []

        for label, category in enumerate(['normal', 'malicious']):
            dir_path = os.path.join(base_path, category)
            for f in os.listdir(dir_path):
                if f.lower().endswith(('.wav', '.mp3')):
                    path = os.path.join(dir_path, f)
                    audio, _ = librosa.load(path, sr=self.sr)
                    print(audio)

                    # Logic: Calculate step based on your 8.5s rule
                    if len(audio) < int(8.5 * sr) and len(audio) >= self.window_size:
                        step = (len(audio) - self.window_size) // 1
                    else:
                        step = self.window_size - self.min_overlap

                    # Sliding Window Loop
                    for start in range(0, len(audio) - self.window_size + 1, step):
                        self.samples.append((path, start, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, start, label = self.samples[idx]
        audio, _ = librosa.load(path, sr=self.sr, offset=start/self.sr, duration=5.0)

        # Spectrogram conversion
        mel_spec = librosa.feature.melspectrogram(y=audio, sr=self.sr, n_mels=128)
        log_mel = librosa.power_to_db(mel_spec, ref=np.max)

        return torch.tensor(log_mel).unsqueeze(0), torch.tensor(label)
p=SpectrogramDataset('dataset/dataset')

ModuleNotFoundError: No module named 'torch'

In [ ]:

class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(32 * 32 * 39, 128), nn.ReLU(), # Shape depends on sr/window
            nn.Linear(128, 2)
        )

    def forward(self, x):
        return self.conv(x)

# Initialization and Splitting
dataset = SpectrogramDataset('dataset/dataset')
train_sz = int(0.8 * len(dataset))
val_sz = int(0.1 * len(dataset))
test_sz = len(dataset) - train_sz - val_sz

train_db, val_db, test_db = random_split(dataset, [train_sz, val_sz, test_sz])

train_loader = DataLoader(train_db, batch_size=16, shuffle=True)
model = SimpleCNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
torch.save(model.state_dict(), 'malicious_call_detector1')

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

Looking in indexes: https://download.pytorch.org/whl/cu118
  Obtaining dependency information for torch from https://download.pytorch.org/whl/cu118/torch-2.7.1%2Bcu118-cp311-cp311-manylinux_2_28_x86_64.whl.metadata
  Obtaining dependency information for torchvision from https://download.pytorch.org/whl/cu118/torchvision-0.22.1%2Bcu118-cp311-cp311-manylinux_2_28_x86_64.whl.metadata
  Obtaining dependency information for torchaudio from https://download.pytorch.org/whl/cu118/torchaudio-2.7.1%2Bcu118-cp311-cp311-manylinux_2_28_x86_64.whl.metadata
  Obtaining dependency information for sympy>=1.13.3 from https://files.pythonhosted.org/packages/a2/09/77d55d46fd61b4a135c444fc97158ef34a095e5681d0a6c10b75bf356191/sympy-1.14.0-py3-none-any.whl.metadata
  Obtaining dependency information for networkx from https://files.pythonhosted.org/packages/9e/c9/b2622292ea83fbb4ec318f5b9ab867d0a28ab43c5717bb85b0a5f6b3b0a4/networkx-3.6.1-py3-none-any.whl.metadata
     ━━━━━━━━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# To load later
model = SimpleCNN()
model.load_state_dict(torch.load('malicious_call_detector1'))
model.eval()